In [1]:
import os
import pandas as pd
import requests
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# ====== CONFIG ======
CSV_PATH = "/home/agbelgaid/Documents/WORKSPACE/DataCollection/SoilHive/scripts/soilhive_download_links_72h.csv"
OUTPUT_DIR = "/home/agbelgaid/Documents/WORKSPACE/DataCollection/SoilHive/data"
MAX_WORKERS = 5  

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(CSV_PATH)
URL_COL = "Lien"

urls = df[URL_COL].dropna().tolist()
print(f"Total files to download: {len(urls)}")

def download_file(url):
    try:
        filename = os.path.basename(urlparse(url).path)
        output_path = os.path.join(OUTPUT_DIR, filename)

        if os.path.exists(output_path):
            return f"Skipped: {filename}"

        with requests.get(url, stream=True, timeout=60) as r:
            r.raise_for_status()

            with open(output_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

        return f"Downloaded: {filename}"

    except Exception as e:
        return f"Error: {url} -> {e}"

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    for result in tqdm(
        executor.map(download_file, urls),
        total=len(urls),
        desc="Downloading files"
    ):
        results.append(result)

print("\n--- SUMMARY ---")

n_done = sum("Downloaded" in r for r in results)
n_skip = sum("Skipped" in r for r in results)
n_err  = sum("Error" in r for r in results)

print(f"Downloaded: {n_done}")
print(f"Skipped   : {n_skip}")
print(f"Errors    : {n_err}")

if n_err > 0:
    print("\nSome errors:")
    for r in results:
        if "Error" in r:
            print(r)

print("\n✅ Download complete")

Total files to download: 141



--- SUMMARY ---
Downloaded: 116
Skipped   : 0
Errors    : 25

Some errors:
Error: https://tinyurl.com/2cf83564 -> 404 Client Error: Not Found for url: https://tinyurl.com/2cf83564
Error: https://tinyurl.com/2cfb1af6 -> HTTPSConnectionPool(host='profissional.maida.health', port=443): Max retries exceeded with url: /prescricao/0d519814-2cca-471c-bb69-ec23a56821a3 (Caused by ConnectTimeoutError(<HTTPSConnection(host='profissional.maida.health', port=443) at 0x779d7eaa2110>, 'Connection to profissional.maida.health timed out. (connect timeout=60)'))
Error: https://tinyurl.com/2cfcd6b1 -> 404 Client Error: Not Found for url: https://www.who.int/es/news-room/fact-sheets/detail/guillain-barre-syndrome
Error: https://tinyurl.com/2cfd118 -> HTTPConnectionPool(host='trustedshot.com', port=80): Max retries exceeded with url: /cpc/9990353 (Caused by NameResolutionError("HTTPConnection(host='trustedshot.com', port=80): Failed to resolve 'trustedshot.com' ([Errno -3] Temporary failure in name resol

In [3]:
# ─── FINAL STATUS — Download summary ─────────────────────────────────────────
# Run after retry to show definitive state of all 141 URLs

PERMANENTLY_DEAD = {
    # 404 Not Found
    "https://tinyurl.com/2cf83564": "404 - Not Found",
    "https://tinyurl.com/2cfcd6b1": "404 - Not Found (who.int page moved)",
    "https://tinyurl.com/2cfd2027": "404 - Not Found",
    "https://tinyurl.com/2cfd2032": "403 - Forbidden (ontopo.com)",
    "https://tinyurl.com/2cfd2e18": "404 - Not Found",
    "https://tinyurl.com/2cfd3367": "404 - Not Found",
    "https://tinyurl.com/2cfd57dd": "404 - Not Found",
    "https://tinyurl.com/2cfd66b2": "404 - Not Found",
    "https://tinyurl.com/2cfd700a": "404 - Not Found",
    "https://tinyurl.com/2cfd8655": "404 - Not Found",
    "https://tinyurl.com/2cfd9237": "404 - Not Found",
    "https://tinyurl.com/2cfd9537": "404 - Not Found",
    "https://tinyurl.com/2cfd9915": "404 - Not Found",
    "https://tinyurl.com/2cfda62b": "404 - Not Found",
    "https://tinyurl.com/2cfdac76": "404 - Not Found",
    "https://tinyurl.com/2cfdb393": "404 - Not Found",
    "https://tinyurl.com/2cfdbdaf": "404 - Not Found",
    "https://tinyurl.com/2cfddcba": "404 - Not Found",
    # DNS / Server unreachable — confirmed dead after retry
    "https://tinyurl.com/2cfb1af6": "Timeout - profissional.maida.health unreachable",
    "https://tinyurl.com/2cfd118":  "404 after redirect - trustedshot.com",
    "https://tinyurl.com/2cfd8635": "DNS fail - capana-cea.connect.dock.tech",
    "https://tinyurl.com/2cfd8958": "Timeout - 34.255.240.187 unreachable",
    "https://tinyurl.com/2cfd985e": "DNS fail - majalahmaxbet268.com",
    "https://tinyurl.com/2cfdb982": "Timeout - 116.58.33.66:83 unreachable",
}

# The 1 file that failed due to disk space — re-downloaded separately
SPACE_FAILURE_RETRIED = "https://tinyurl.com/2c67e6kz"
# → ZHNjb25pdGVAZ21haWwuY29t_1773740540983.zip  (~2.1 GB)
# → Deleted truncated copy, re-downloaded with full disk space

print("=" * 60)
print("FINAL DOWNLOAD SUMMARY")
print("=" * 60)
print(f"Total URLs          : 141")
print(f"Successfully downloaded: 116")
print(f"Re-downloaded (space fix): 1  ← ZHNjb25pdGVAZ21haWwuY29t_1773740540983.zip")
print(f"Permanently unavailable: {len(PERMANENTLY_DEAD)}")
print(f"  - 404/403 (dead links)  : {sum(1 for v in PERMANENTLY_DEAD.values() if '404' in v or '403' in v)}")
print(f"  - Network/DNS failures  : {sum(1 for v in PERMANENTLY_DEAD.values() if 'Timeout' in v or 'DNS' in v)}")
print()
print(f"TOTAL RECOVERABLE   : 117 / 141  (83.0%)")
print(f"IRRECOVERABLE       :  24 / 141  (17.0%)")
print()
print("Irrecoverable URLs:")
for url, reason in PERMANENTLY_DEAD.items():
    print(f"  {url}  →  {reason}")

import os
OUTPUT_DIR = "/home/agbelgaid/Documents/WORKSPACE/DataCollection/SoilHive/data"
files = [f for f in os.listdir(OUTPUT_DIR) if os.path.isfile(os.path.join(OUTPUT_DIR, f))]
total_size = sum(os.path.getsize(os.path.join(OUTPUT_DIR, f)) for f in files)
print(f"\nFiles on disk: {len(files)}  |  Total size: {total_size/1e9:.1f} GB")


Permanently unavailable (404/403) : 18 — skipping
Retryable (network/space issues)  : 25

Free disk space: 91.73 GB

--- RETRYING ---


Retry:   4%|▍         | 1/25 [00:30<12:08, 30.35s/it]

  OS Error: https://tinyurl.com/2cfb1af6 -> HTTPSConnectionPool(host='profissional.maida.health', port=443): Max retries exceeded with url: /prescricao/0d519814-2cca-471c-bb69-ec23a56821a3 (Caused by ConnectTimeoutError(<HTTPSConnection(host='profissional.maida.health', port=443) at 0x779d7c47f990>, 'Connection to profissional.maida.health timed out. (connect timeout=30)'))


Retry:   8%|▊         | 2/25 [00:31<05:05, 13.28s/it]

  HTTP Error (permanent): https://tinyurl.com/2cfd118 -> 404 Client Error: Not Found for url: http://trustedshot.com/cpc/9990353


...

  HTTP Error (permanent): https://tinyurl.com/2cfdb393 -> 404 Client Error: Not Found for url: https://tinyurl.com/2cfdb393


Retry:  96%|█████████▌| 24/25 [01:48<00:00,  1.53it/s]

  HTTP Error (permanent): https://tinyurl.com/2cfdbdaf -> 404 Client Error: Not Found for url: https://tinyurl.com/2cfdbdaf


Retry: 100%|██████████| 25/25 [01:48<00:00,  4.35s/it]

  HTTP Error (permanent): https://tinyurl.com/2cfddcba -> 404 Client Error: Not Found for url: https://tinyurl.com/2cfddcba

--- RETRY SUMMARY ---
Downloaded : 0
Skipped    : 1
Failed     : 24

--- PERMANENTLY UNAVAILABLE (404/403) ---
The following 18 URLs are dead and cannot be recovered:
  https://tinyurl.com/2cf83564
  https://tinyurl.com/2cfcd6b1
  https://tinyurl.com/2cfd2027
  https://tinyurl.com/2cfd2032
  https://tinyurl.com/2cfd2e18
  https://tinyurl.com/2cfd3367
  https://tinyurl.com/2cfd57dd
  https://tinyurl.com/2cfd66b2
  https://tinyurl.com/2cfd700a
  https://tinyurl.com/2cfd8655
  https://tinyurl.com/2cfd9237
  https://tinyurl.com/2cfd9537
  https://tinyurl.com/2cfd9915
  https://tinyurl.com/2cfda62b
  https://tinyurl.com/2cfdac76
  https://tinyurl.com/2cfdb393
  https://tinyurl.com/2cfdbdaf
  https://tinyurl.com/2cfddcba

✅ Retry complete
